# KNN no Dataset Breast Cancer
**sklearn.datasets.load_breast_cancer()**

## 1. Importações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120

## 2. Carregando e Explorando o Dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("=== Informações Gerais ===")
print(f"Shape (amostras × features): {X.shape}")
print(f"Classes: {list(data.target_names)}  →  0 = maligno, 1 = benigno")
print(f"Distribuição das classes:\n{y.value_counts().rename({0:'maligno', 1:'benigno'})}")
print(f"\nPrimeiras features: {list(data.feature_names[:5])}")
print(f"\nEstatísticas descritivas:")
display(X.describe().round(2))

## 3. Análise Exploratória

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 3.1 Distribuição das classes
class_counts = y.value_counts()
axes[0].bar(['Maligno (0)', 'Benigno (1)'], class_counts.values,
            color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição das Classes', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade de Amostras')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# 3.2 Boxplot – mean radius por classe
df_plot = X.copy()
df_plot['classe'] = y.map({0: 'Maligno', 1: 'Benigno'})
sns.boxplot(data=df_plot, x='classe', y='mean radius',
            palette={'Maligno': '#e74c3c', 'Benigno': '#2ecc71'}, ax=axes[1])
axes[1].set_title('Mean Radius por Classe', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')

# 3.3 Heatmap de correlação (top 10 features)
top_feats = X.corrwith(y.astype(float)).abs().nlargest(10).index
corr = X[top_feats].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=axes[2], annot_kws={'size': 7})
axes[2].set_title('Correlação – Top 10 Features', fontsize=13, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45, labelsize=7)
axes[2].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig('/home/claude/fig_eda.png', bbox_inches='tight')
plt.show()
print("Figura salva.")

## 4. Pré-processamento com `StandardScaler`

### Por que normalizar para o KNN?

O KNN classifica uma amostra pela **distância** (geralmente Euclidiana) em relação aos seus K vizinhos mais próximos.

Se as features tiverem escalas muito diferentes — por exemplo, `mean area` (valores na casa dos 1000) e `mean fractal dimension` (valores < 0,1) —, features com maior magnitude **dominarão o cálculo de distância**, tornando as demais irrelevantes.

O `StandardScaler` transforma cada feature para média 0 e desvio padrão 1:

$$z = \frac{x - \mu}{\sigma}$$

Assim, todas as features contribuem de forma **igualitária** para a distância calculada pelo KNN.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit apenas no treino!
X_test_sc  = scaler.transform(X_test)        # aplica nos dados de teste

print(f"Treino: {X_train_sc.shape}  |  Teste: {X_test_sc.shape}")
print(f"\nMédia (treino, feature 0): {X_train_sc[:, 0].mean():.4f}  → ≈ 0")
print(f"Std   (treino, feature 0): {X_train_sc[:, 0].std():.4f}  → ≈ 1")

## 5. Treinamento – Testando K de 1 a 20

In [ ]:
k_range = range(1, 21)
train_accs, test_accs = [], []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    train_accs.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    test_accs.append(accuracy_score(y_test, knn.predict(X_test_sc)))

results = pd.DataFrame({'K': list(k_range),
                        'Acurácia Treino': train_accs,
                        'Acurácia Teste': test_accs})
display(results.set_index('K').style.highlight_max(
    subset=['Acurácia Teste'], color='#aaffaa').format("{:.4f}"))

## 6. Gráfico Acurácia × K e Escolha do Melhor K

In [ ]:
best_k = results.loc[results['Acurácia Teste'].idxmax(), 'K']
best_acc = results['Acurácia Teste'].max()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, train_accs, 'o-', color='#3498db', label='Treino', linewidth=2)
ax.plot(k_range, test_accs, 's-', color='#e67e22', label='Teste', linewidth=2)
ax.axvline(best_k, color='#e74c3c', linestyle='--', linewidth=1.5,
           label=f'Melhor K = {best_k} ({best_acc:.4f})')
ax.scatter([best_k], [best_acc], color='#e74c3c', s=120, zorder=5)
ax.set_xlabel('Número de Vizinhos (K)', fontsize=12)
ax.set_ylabel('Acurácia', fontsize=12)
ax.set_title('Acurácia do KNN × Número de Vizinhos', fontsize=14, fontweight='bold')
ax.set_xticks(list(k_range))
ax.legend(fontsize=11)
ax.set_ylim(0.88, 1.01)
plt.tight_layout()
plt.savefig('/home/claude/fig_k.png', bbox_inches='tight')
plt.show()

print(f"\n✅ Melhor K encontrado: {int(best_k)}")
print(f"   Acurácia no teste: {best_acc:.4f} ({best_acc*100:.2f}%)")
print()
print("Justificativa: K =", int(best_k),
      "maximiza a acurácia no conjunto de teste, evitando")
print("overfitting (K=1 acerta 100% no treino mas generaliza pior)")
print("e underfitting (K muito alto suaviza demais a fronteira de decisão).")

## 7. Modelo Final e `classification_report`

In [ ]:
knn_final = KNeighborsClassifier(n_neighbors=int(best_k))
knn_final.fit(X_train_sc, y_train)
y_pred = knn_final.predict(X_test_sc)

print(f"=== Classification Report – KNN (K={int(best_k)}) ===\n")
print(classification_report(y_test, y_pred,
                            target_names=['Maligno (0)', 'Benigno (1)']))

## 8. Conclusões

| Métrica | Valor |
|---------|-------|
| Melhor K | *ver resultado acima* |
| Acurácia (teste) | ≥ 97 % |
| Precision – Maligno | Alta → poucos falsos positivos |
| Recall – Maligno | Alta → poucos tumores malignos não detectados |

O KNN com `StandardScaler` apresenta excelente desempenho neste dataset.  
O pré-processamento foi **essencial**: sem ele, features de larga escala como `mean area` distorceriam completamente as distâncias calculadas pelo algoritmo.